# G2：ROS 2 服务调用与参数 —— 请求/响应与配置管理# G2: ROS 2 Services & Parameters — Request/Response and Configuration## 一、实验原理 | Principle### 1.1 服务（Service）vs 话题（Topic）### 1.1 Service vs TopicG1 讲了话题（异步发布/订阅）。服务是 ROS 2 的第二种通信模式：**同步请求/响应**。G1 covered topics (async publish/subscribe). Services are the second ROS 2 communication pattern: **synchronous request/response**.| 对比维度 \| Dimension | 话题 Topic | 服务 Service ||---|---|---|| 通信模式 \| Pattern | 发布/订阅 publish/subscribe | 请求/响应 request/response || 时序 \| Timing | 异步 async | 同步（阻塞等应答） sync (blocks for reply) || 典型用途 \| Use case | 高频状态流，如 `/vehicle/speed` | 低频查询/命令，如 `/aeb/set_enabled` || 类比 \| Analogy | 广播电台 radio broadcast | 打电话 phone call |### 1.2 参数服务器（Parameter Server）### 1.2 Parameter Server每个 ROS 2 节点自带参数服务器：**键值配置**（`declare_parameter` 声明，`get_parameter` 读取，`set_parameters` 修改）。Every ROS 2 node has a built-in parameter server: **key-value configuration** (`declare_parameter` to declare, `get_parameter` to read, `set_parameters` to update).典型用途 Typical uses：速度上限 `max_speed`、TTC 阈值 `aeb_ttc_threshold`（对应 A1 实验的调参）、车辆编号 `vehicle_name`。### 1.3 在 IEV CloudLab 中的定位### 1.3 Role in IEV CloudLab> **重要 Important**：本平台中 ROS 2 **不做车辆闭环控制**！> A1/B3 的控制器直调仿真器 API（延迟 <1ms），走 ROS 2 会有 5–10ms 延迟，不适合实时控制。>> ROS 2 用于：**参数服务**（如在线调 TTC 阈值）、**多车状态广播**、**数据记录**（`foxglove_bridge` 可视化）。>> In this platform ROS 2 is **NOT used for vehicle closed-loop control**! A1/B3 controllers call the simulator API directly (<1ms latency); going through ROS 2 adds 5–10ms, unsuitable for real-time control.>> ROS 2 is used for: **parameter services** (e.g. tuning the TTC threshold live), **multi-vehicle state broadcast**, **data logging** (`foxglove_bridge` visualization).## 二、实验目的 | Objectives1. 理解服务与话题的适用场景差异 | Understand when to use services vs topics2. 掌握服务的请求/响应流程（server + client） | Master the service request/response flow3. 会用参数服务器做配置管理 | Use the parameter server for configuration4. 明确 ROS 2 在本平台中的定位 | Know ROS 2's role in this platform## 三、学生任务 | Your Tasks1. **运行 Run**：执行下面的真实 ROS 2 演示（服务开关 AEB + 参数读写）2. **修改 Modify**：把 `aeb_ttc_threshold` 改成 4.0，观察参数变化3. **思考 Think**：为什么“开关 AEB”适合用服务，而“车辆速度”适合用话题？   Why is "toggling AEB" a good fit for services, while "vehicle speed" fits topics?## 四、验证内容 | Verification- [ ] 服务调用返回正确结果（AEB enabled/disabled）| Service calls return correct results- [ ] 参数读写成功（declare/get/set）| Parameter read/write works- [ ] 理解服务 vs 话题的选型 | Can choose between service and topic- [ ] 理解 ROS 2 不做车辆闭环的原因 | Understand why ROS 2 is excluded from closed-loop control

## 1. 环境检查与演示代码## 1. Environment Check & Demo Code真实 ROS 2 Jazzy 位于 `/opt/ros/jazzy`。下面先检查环境，然后把演示脚本写入 `/tmp`。Real ROS 2 Jazzy lives at `/opt/ros/jazzy`. Check the environment first, then write the demo script to `/tmp`.> 说明 Note：C 扩展的库路径（`LD_LIBRARY_PATH`）必须在 Python 进程**启动前**就位，notebook 内无法直接 `import rclpy`，因此演示脚本在正确 source 环境的子进程中运行——等价于你在终端执行 `source /opt/ros/jazzy/setup.bash && python3 demo.py`，是**真实的 ROS 2 运行**，不是模拟。>> The native library path must exist **before** the Python process starts, so `import rclpy` cannot workdirectly inside the notebook kernel. The demo therefore runs in a subprocess with the properly sourcedenvironment — equivalent to `source /opt/ros/jazzy/setup.bash && python3 demo.py` in a terminal.This is **real ROS 2 execution**, not a simulation.

In [ ]:
ROS_DEMO_SCRIPT = r'''
"""G2 real ROS 2 demo: service server/client + parameter server.

Runs under /opt/ros/jazzy (sourced env). Single-threaded: rclpy.spin_once()
pumps both server callbacks and client responses, so shutdown is clean.
"""
import os
os.environ.setdefault('ROS_DOMAIN_ID', '42')
import time
import rclpy
from rclpy.node import Node
from rclpy.parameter import Parameter
from std_srvs.srv import SetBool


def spin_until(node, future, timeout=5.0):
    """Pump the executor until the future completes or timeout."""
    t0 = time.time()
    while not future.done() and time.time() - t0 < timeout:
        rclpy.spin_once(node, timeout_sec=0.1)
    assert future.done(), 'service call timed out'
    return future.result()


def main():
    rclpy.init()
    node = Node('g2_service_demo')
    state = {'aeb_enabled': False}

    # ---- Service server: /aeb/set_enabled ----
    def set_aeb_callback(request, response):
        state['aeb_enabled'] = bool(request.data)
        response.success = True
        response.message = 'AEB enabled' if request.data else 'AEB disabled'
        return response
    node.create_service(SetBool, 'aeb/set_enabled', set_aeb_callback)
    print('[server] service /aeb/set_enabled ready', flush=True)

    # ---- Parameter server: declare defaults ----
    node.declare_parameter('max_speed', 15.0)          # m/s speed limit
    node.declare_parameter('vehicle_name', 'ego')      # vehicle id string
    node.declare_parameter('aeb_ttc_threshold', 3.0)   # s, links to A1 experiment

    # ---- Service client ----
    client = node.create_client(SetBool, 'aeb/set_enabled')
    assert client.wait_for_service(timeout_sec=5.0), 'service not available'

    req = SetBool.Request(); req.data = True
    res = spin_until(node, client.call_async(req))
    print(f"[client] set_enabled(True) -> success={res.success}, message='{res.message}'", flush=True)
    assert res.success and res.message == 'AEB enabled' and state['aeb_enabled'] is True

    req = SetBool.Request(); req.data = False
    res = spin_until(node, client.call_async(req))
    print(f"[client] set_enabled(False) -> success={res.success}, message='{res.message}'", flush=True)
    assert res.success and res.message == 'AEB disabled' and state['aeb_enabled'] is False

    # ---- Parameter server: read / write ----
    print(f"[param] max_speed = {node.get_parameter('max_speed').value}", flush=True)
    print(f"[param] vehicle_name = {node.get_parameter('vehicle_name').value}", flush=True)
    print(f"[param] aeb_ttc_threshold = {node.get_parameter('aeb_ttc_threshold').value}", flush=True)
    node.set_parameters([Parameter('max_speed', Parameter.Type.DOUBLE, 25.0)])
    node.set_parameters([Parameter('aeb_ttc_threshold', Parameter.Type.DOUBLE, 2.0)])
    assert node.get_parameter('max_speed').value == 25.0
    assert node.get_parameter('aeb_ttc_threshold').value == 2.0
    print('[param] max_speed after set = 25.0', flush=True)
    print('[param] aeb_ttc_threshold after set = 2.0', flush=True)

    node.destroy_node()
    rclpy.shutdown()
    print('G2_DEMO_PASS', flush=True)


if __name__ == '__main__':
    main()

'''

import os
# Check that the real ROS 2 Jazzy installation exists in this container
assert os.path.exists('/opt/ros/jazzy'), 'ROS 2 Jazzy not found at /opt/ros/jazzy!'
assert os.path.exists('/opt/ros/jazzy/setup.bash'), 'setup.bash missing!'
print('ROS 2 Jazzy found:', os.path.exists('/opt/ros/jazzy'))

# Write the demo script to /tmp so the sourced subprocess can run it
with open('/tmp/g2_service_demo.py', 'w') as f:
    f.write(ROS_DEMO_SCRIPT)
print('Demo script written to /tmp/g2_service_demo.py')
print('Script preview (first 12 lines):')
print('\n'.join(ROS_DEMO_SCRIPT.split('\n')[:12]))


## 2. 运行真实 ROS 2 演示：服务 + 参数## 2. Run the Real ROS 2 Demo: Services + Parameters演示内容 What the demo does：- **服务 Service**：`/aeb/set_enabled`（`std_srvs/srv/SetBool`）——客户端发送 `True`/`False`，服务端切换 AEB 开关并返回确认- **参数 Parameters**：声明 `max_speed` / `vehicle_name` / `aeb_ttc_threshold`（关联 A1 实验），演示读取与在线修改

In [ ]:
import subprocess
# Run the demo in a subprocess with the fully sourced ROS 2 Jazzy environment.
# This executes REAL rclpy: a service server, a service client, and the
# parameter server, all inside this container.
result = subprocess.run(
    ['bash', '-c', 'source /opt/ros/jazzy/setup.bash && timeout 30 python3 /tmp/g2_service_demo.py'],
    capture_output=True, text=True, timeout=40)
# Show the real node output (filter out ROS info-line prefixes for readability)
for line in result.stdout.splitlines():
    print(line)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
# The demo script asserts every step internally; G2_DEMO_PASS means all green.
assert 'G2_DEMO_PASS' in result.stdout, 'ROS 2 demo did not pass!'
assert result.returncode == 0, f'demo exited with code {result.returncode}'
print('\n✅ 真实 ROS 2 服务调用与参数读写验证通过！| Real ROS 2 services & parameters verified!')


## 五、结果分析 | 5. Results Analysis

### 5.1 结果概述 | Overview
| 检查项 | 结果 |
|---|---|
| 服务 `/aeb/set_enabled` 调用 | ✅ 成功（True→"AEB enabled"，False→"AEB disabled"） |
| 参数读取（max_speed 等 3 个） | ✅ 成功 |
| 参数在线修改 | ✅ 成功（25.0 / 2.0 断言通过） |
| 演示脚本退出码 | ✅ 0（`G2_DEMO_PASS`） |

### 5.2 服务调用可靠性 | Service Reliability
两次服务调用（开/关 AEB）均一次成功，返回的 `success` 标志与 `message` 字符串均符合预期。服务的**同步请求/响应**语义得到验证：客户端阻塞等待，直到服务端处理完成——这与话题的"发后不管"形成鲜明对比，也是服务适合**配置类操作**（开关功能、查询状态）的原因。

### 5.3 参数服务器的作用 | Parameter Server
`max_speed`、`vehicle_name`、`aeb_ttc_threshold` 三个参数演示了参数服务器的典型用法：
- **启动配置**：节点启动时声明默认值；
- **在线调参**：运行时修改 `aeb_ttc_threshold`（2.0s）无需重启节点——这正是 A1 实验中"调阈值看效果"的工程对应物；
- **类型安全**：参数有类型约束，类型不匹配会报错（见报错表）。

### 5.4 与 G1 的对比 | vs G1 Topics
| 维度 | 话题（G1） | 服务（本实验） |
|---|---|---|
| 模式 | 异步发布/订阅 | 同步请求/响应 |
| 适用 | 高频状态流 | 低频配置/查询 |
| 可靠性 | Best-Effort 可丢包 | 必须应答 |
| 本平台用途 | 多车状态广播 | AEB 开关、阈值配置 |

两者互补，共同构成 ROS 2 在 IEV CloudLab 中的**非实时通信层**定位。


### 常见报错说明 | Troubleshooting| 报错信息 \| Error | 中文解释 \| Explanation | 解决方法 \| Fix ||---|---|---|| `ROS 2 Jazzy not found` | 容器里没有 `/opt/ros/jazzy` | 确认镜像为 `iev-base:jazzy-24.04`，重启容器 || `service not available` | 服务端 5 秒内未就绪 | 检查 `ROS_DOMAIN_ID` 是否一致（本实验用 42） || `service call timed out` | 请求发出后无应答 | `spin_once` 未被调用，检查执行器循环 || `ModuleNotFoundError: rclpy` | 在 notebook 内直接 import | 必须用 source 环境的子进程运行（见说明） || `librcl_*.so: cannot open` | C 扩展库路径缺失 | 同上：库路径必须在进程启动前设置 || `ParameterType` 不匹配 | set 时类型与声明不一致 | `declare_parameter('max_speed', 15.0)` 是 DOUBLE，set 时也要用 DOUBLE |## 🏆 挑战任务 | Challenge1. 在终端用命令行调服务：在 JupyterLab 打开 Terminal，执行   `source /opt/ros/jazzy/setup.bash`，然后 `ros2 service list` 查看服务（需先另起终端跑 demo 常驻版）。2. 设计一个 `/platoon/set_gap` 服务：用 `std_srvs/srv/SetBool` 或查 `ros2 interface show` 找更合适的类型，   为 B3 的 CACC 车队设计“切换跟驰模式”服务。3. 思考：`aeb_ttc_threshold` 如果通过参数服务在线修改，A1 的 AEB 控制器要如何配合？（提示：控制循环中定期 `get_parameter`）

---

## 下一个实验 | Next →

**[H1_batch_generation](./H1_batch_generation.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
